# Демо проектных упражнений модуля 17 — Кандидаты, стоимость реранкинга и позднее разбиение

**Демо-реализация, не готовое решение полномасштабного проекта.** Демо 1 выполняет BM25, LSA, RRF и реранкинг на четырёх вымышленных записях, включая похожие коды ошибок. Все оценки сохраняются; стоимость измеряется числом реально оценённых пар. Демо 2 выполняет внимание до и после разбиения на специально заданных векторах и показывает перенос сигнала из соседнего фрагмента.
Все входы малы, детерминированы и работают без сети на CPU. Цель — проверить
интерфейсы, метрики, инварианты и структуру отчёта до расходования GPU/API-бюджета.

In [ ]:
from __future__ import annotations
import importlib.util, subprocess, sys
for package in ("numpy", "matplotlib"):
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])


import math
import random
import time
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import numpy as np

SEED = 20260801
random.seed(SEED)
np.random.seed(SEED)
print(f"воспроизводимый тестовый запуск; seed={SEED}")
DEMO_LANGUAGE = 'ru'


В первом опыте меняется только глубина переоценки: один, два или четыре кандидата. Сначала проверьте, в каком месте каждого списка оказался error999. Затем выясните, доступен ли он реранкеру при каждой глубине. Число оценённых пар описывает работу, а не секунды: пары можно обрабатывать пакетами. Все варианты используют один и тот же запрос; качество на новых запросах здесь не измеряется.

In [ ]:
import math, re, json
from collections import Counter
from dataclasses import dataclass
from typing import Mapping, Sequence
WORD_RE = re.compile(r"[^\W_]+", re.UNICODE)
def _positive_int(name: str, value: int) -> int:
    """Validate and return one strictly positive integer."""

    if not isinstance(value, (int, np.integer)) or isinstance(value, bool) or int(value) <= 0:
        raise ValueError(f"{name} must be a positive integer")
    return int(value)

def _positive_finite(name: str, value: float) -> float:
    """Validate and return one strictly positive finite float."""

    value = float(value)
    if not math.isfinite(value) or value <= 0.0:
        raise ValueError(f"{name} must be finite and positive")
    return value

def tokenize(text: str) -> tuple[str, ...]:
    """Find word spans in the original text, then casefold each word for search."""

    if not isinstance(text, str):
        raise TypeError("text must be a string")
    return tuple(match.group().casefold() for match in WORD_RE.finditer(text))

def _validate_ids(ids):
    """IDs remain nonempty strings; coercion must not merge 1 and '1'."""
    if any(not isinstance(i, str) or not i.strip() for i in ids):
        raise ValueError("document IDs must be nonempty strings")
    if len(ids) != len(set(ids)):
        raise ValueError("document IDs must be unique")

def _validate_judgments(ranking, relevance):
    """Reject duplicate ranks, fractional/negative grades and empty gold sets."""
    _validate_ids(ranking)
    _validate_ids(list(relevance))
    if any(isinstance(g, bool) or not isinstance(g, (int, np.integer))
           or not 0 <= g <= 30 for g in relevance.values()):
        raise ValueError("grades must be integers from 0 through 30")
    if not any(g > 0 for g in relevance.values()):
        raise ValueError("at least one positive judgment is required")

def stable_ranking(scores: Sequence[float], document_ids: Sequence[str]) -> tuple[str, ...]:
    """Sort scores descending with document ID as a deterministic tie-break."""

    score_array = np.asarray(scores, dtype=np.float64)
    if score_array.ndim != 1 or score_array.size != len(document_ids):
        raise ValueError("scores and document_ids must be aligned rank-1 sequences")
    if not np.all(np.isfinite(score_array)):
        raise ValueError("scores contain NaN or infinity")
    _validate_ids(document_ids)
    if len(set(document_ids)) != len(document_ids):
        raise ValueError("document_ids must be unique")
    return tuple(
        document_id
        for _, document_id in sorted(
            zip(score_array.tolist(), map(str, document_ids)),
            key=lambda pair: (-pair[0], pair[1]),
        )
    )

@dataclass(frozen=True)
class SearchResult:
    """One ranked document and its branch score."""

    document_id: str
    score: float
    rank: int

class BM25Index:
    """A small exact BM25 index with explicit empty-input policy."""

    def __init__(
        self,
        document_ids: Sequence[str],
        texts: Sequence[str],
        *,
        k1: float = 1.5,
        b: float = 0.75,
    ) -> None:
        if len(document_ids) != len(texts) or not document_ids:
            raise ValueError("document_ids and texts must be equally sized and non-empty")
        _validate_ids(document_ids)
        self.document_ids = tuple(map(str, document_ids))
        self.tokens = tuple(tokenize(text) for text in texts)
        if any(not tokens for tokens in self.tokens):
            raise ValueError("empty documents must be removed before indexing")
        self.k1 = _positive_finite("k1", k1)
        self.b = float(b)
        if not math.isfinite(self.b) or not 0.0 <= self.b <= 1.0:
            raise ValueError("b must lie in [0,1]")
        n_documents = len(self.tokens)
        self.average_length = sum(map(len, self.tokens)) / n_documents
        self.document_frequency = Counter(
            token for document_tokens in self.tokens for token in set(document_tokens)
        )
        self.idf = {
            token: math.log(
                (n_documents - frequency + 0.5) / (frequency + 0.5) + 1.0
            )
            for token, frequency in self.document_frequency.items()
        }

    def scores(self, query: str) -> np.ndarray:
        """Return one BM25 score per indexed document."""

        query_tokens = tokenize(query)
        if not query_tokens:
            raise ValueError("query must contain at least one word token")
        if not any(token in self.idf for token in query_tokens):
            raise ValueError("query has no indexed lexical terms")
        scores = np.zeros(len(self.tokens), dtype=np.float64)
        for row, document_tokens in enumerate(self.tokens):
            frequencies = Counter(document_tokens)
            length_norm = 1.0 - self.b + self.b * len(document_tokens) / self.average_length
            for token in query_tokens:
                tf = frequencies.get(token, 0)
                if tf == 0 or token not in self.idf:
                    continue
                scores[row] += (
                    self.idf[token]
                    * tf
                    * (self.k1 + 1.0)
                    / (tf + self.k1 * length_norm)
                )
        return scores

    def search(self, query: str, *, top_k: int | None = None) -> tuple[SearchResult, ...]:
        """Rank all documents or the requested top-k with stable ties."""

        scores = self.scores(query)
        ranking = stable_ranking(scores, self.document_ids)
        if top_k is not None:
            top_k = _positive_int("top_k", top_k)
            ranking = ranking[:top_k]
        score_by_id = dict(zip(self.document_ids, scores.tolist()))
        return tuple(
            SearchResult(document_id, float(score_by_id[document_id]), rank)
            for rank, document_id in enumerate(ranking, start=1)
        )

class LSAIndex:
    """Exact cosine search over dense TF-IDF + SVD document vectors."""

    def __init__(self, document_ids: Sequence[str], texts: Sequence[str], *, dimension: int = 4) -> None:
        if len(document_ids) != len(texts) or not document_ids:
            raise ValueError("document_ids and texts must be equally sized and non-empty")
        _validate_ids(document_ids)
        self.document_ids = tuple(map(str, document_ids))
        tokenized = tuple(tokenize(text) for text in texts)
        if any(not tokens for tokens in tokenized):
            raise ValueError("empty documents must be removed before indexing")
        vocabulary = sorted({token for tokens in tokenized for token in tokens})
        self.vocabulary = {token: index for index, token in enumerate(vocabulary)}
        counts = np.zeros((len(tokenized), len(vocabulary)), dtype=np.float64)
        document_frequency = np.zeros(len(vocabulary), dtype=np.float64)
        for row, tokens in enumerate(tokenized):
            frequencies = Counter(tokens)
            for token, count in frequencies.items():
                counts[row, self.vocabulary[token]] = count
                document_frequency[self.vocabulary[token]] += 1.0
        self.idf = np.log((1.0 + len(tokenized)) / (1.0 + document_frequency)) + 1.0
        tf = np.where(counts > 0.0, 1.0 + np.log(np.maximum(counts, 1.0)), 0.0)
        tfidf = tf * self.idf
        _, singular_values, vt = np.linalg.svd(tfidf, full_matrices=False)
        numerical_rank = int(np.sum(singular_values > 1e-10))
        dimension = min(_positive_int("dimension", dimension), numerical_rank)
        self.projection = vt[:dimension].T
        dense = tfidf @ self.projection
        norms = np.linalg.norm(dense, axis=1, keepdims=True)
        if np.any(norms <= 0.0):
            raise ValueError("at least one document has a zero dense representation")
        self.document_vectors = dense / norms

    def encode_query(self, query: str) -> np.ndarray:
        """Map one non-empty in-vocabulary query to a unit dense vector."""

        query_tokens = tokenize(query)
        if not query_tokens:
            raise ValueError("query must contain at least one word token")
        counts = Counter(query_tokens)
        sparse = np.zeros(len(self.vocabulary), dtype=np.float64)
        for token, count in counts.items():
            position = self.vocabulary.get(token)
            if position is not None:
                sparse[position] = (1.0 + math.log(count)) * self.idf[position]
        dense = sparse @ self.projection
        norm = float(np.linalg.norm(dense))
        if norm <= 0.0:
            raise ValueError("query contains no terms represented by the dense index")
        return dense / norm

    def scores(self, query: str) -> np.ndarray:
        """Return exact cosine scores against every document vector."""

        scores = self.document_vectors @ self.encode_query(query)
        if not np.all(np.isfinite(scores)):
            raise FloatingPointError("dense search produced NaN or infinity")
        return scores

    def search(self, query: str, *, top_k: int | None = None) -> tuple[SearchResult, ...]:
        """Rank exact dense cosine scores with stable ties."""

        scores = self.scores(query)
        ranking = stable_ranking(scores, self.document_ids)
        if top_k is not None:
            top_k = _positive_int("top_k", top_k)
            ranking = ranking[:top_k]
        score_by_id = dict(zip(self.document_ids, scores.tolist()))
        return tuple(
            SearchResult(document_id, float(score_by_id[document_id]), rank)
            for rank, document_id in enumerate(ranking, start=1)
        )

@dataclass(frozen=True)
class FusedResult:
    """One RRF result with a per-branch contribution trace."""

    document_id: str
    score: float
    rank: int
    contributions: Mapping[str, float]
    source_ranks: Mapping[str, int]

def reciprocal_rank_fusion(
    rankings: Mapping[str, Sequence[str]],
    *,
    k: float = 60.0,
    weights: Mapping[str, float] | None = None,
    depth: int | None = None,
) -> tuple[FusedResult, ...]:
    """Accumulate ranks in linear time, then sort unique candidates.

    Ranks are one-based. Duplicate document IDs inside one branch are rejected because
    they would double-count evidence.  The implementation builds rank maps once and
    never calls ``list.index`` inside a loop.
    """

    k = _positive_finite("k", k)
    if not rankings:
        raise ValueError("at least one ranking is required")
    if depth is not None:
        depth = _positive_int("depth", depth)
    branch_weights = {name: 1.0 for name in rankings}
    if weights is not None:
        if set(weights) != set(rankings):
            raise ValueError("weights must contain exactly the ranking branch names")
        branch_weights = {
            name: _positive_finite(f"weight[{name}]", weights[name]) for name in rankings
        }
    scores: dict[str, float] = {}
    contributions: dict[str, dict[str, float]] = {}
    source_ranks: dict[str, dict[str, int]] = {}
    for branch_name, ranking in rankings.items():
        _validate_ids(ranking)
        sequence = tuple(ranking)
        if len(sequence) != len(set(sequence)):
            raise ValueError(f"ranking {branch_name!r} contains duplicate document IDs")
        if depth is not None:
            sequence = sequence[:depth]
        for rank, document_id in enumerate(sequence, start=1):
            contribution = branch_weights[branch_name] / (k + rank)
            scores[document_id] = scores.get(document_id, 0.0) + contribution
            contributions.setdefault(document_id, {})[branch_name] = contribution
            source_ranks.setdefault(document_id, {})[branch_name] = rank
    ordered = sorted(scores, key=lambda document_id: (-scores[document_id], document_id))
    return tuple(
        FusedResult(
            document_id=document_id,
            score=float(scores[document_id]),
            rank=rank,
            contributions=dict(sorted(contributions[document_id].items())),
            source_ranks=dict(sorted(source_ranks[document_id].items())),
        )
        for rank, document_id in enumerate(ordered, start=1)
    )

def _character_trigrams(token: str) -> frozenset[str]:
    """Return boundary-aware character trigrams for one token."""

    wrapped = f"^{token}$"
    return frozenset(wrapped[index : index + 3] for index in range(max(0, len(wrapped) - 2)))

def _token_similarity(left: str, right: str) -> float:
    """Return Jaccard similarity between two token trigram sets."""

    a = _character_trigrams(left)
    b = _character_trigrams(right)
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)

def token_interaction_score(query: str, document: str) -> float:
    """Score a query/document pair with a transparent MaxSim-style proxy.

    This is not a neural cross-encoder.  It deliberately exposes the per-query-token
    maximum interaction used by late-interaction retrieval families.
    """

    query_tokens = tokenize(query)
    document_tokens = tokenize(document)
    if not query_tokens:
        raise ValueError("query must contain at least one word token")
    if not document_tokens:
        raise ValueError("document must contain at least one word token")
    return float(
        sum(
            max(_token_similarity(query_token, document_token) for document_token in document_tokens)
            for query_token in query_tokens
        )
        / len(query_tokens)
    )

def rerank_candidates(
    query: str,
    candidate_ids: Sequence[str],
    documents: Mapping[str, str],
) -> tuple[SearchResult, ...]:
    """Rerank a candidate list with the token-interaction proxy."""

    _validate_ids(candidate_ids)
    candidate_ids = tuple(candidate_ids)
    if not candidate_ids:
        raise ValueError("candidate_ids must be non-empty")
    if len(candidate_ids) != len(set(candidate_ids)):
        raise ValueError("candidate_ids must be unique")
    missing = [document_id for document_id in candidate_ids if document_id not in documents]
    if missing:
        raise KeyError(f"missing documents: {missing}")
    scores = [token_interaction_score(query, documents[document_id]) for document_id in candidate_ids]
    ranking = stable_ranking(scores, candidate_ids)
    score_by_id = dict(zip(candidate_ids, scores))
    return tuple(
        SearchResult(document_id, float(score_by_id[document_id]), rank)
        for rank, document_id in enumerate(ranking, start=1)
    )

def reciprocal_rank(ranking: Sequence[str], relevance: Mapping[str, int]) -> float:
    """Return reciprocal rank of the first positively relevant document."""
    _validate_judgments(ranking, relevance)

    if not any(int(grade) > 0 for grade in relevance.values()):
        raise ValueError("relevance must contain at least one positive grade")
    for rank, document_id in enumerate(ranking, start=1):
        if int(relevance.get(document_id, 0)) > 0:
            return 1.0 / rank
    return 0.0

def recall_at_k(ranking: Sequence[str], relevance: Mapping[str, int], k: int) -> float:
    """Return set recall among all positively relevant documents."""
    _validate_judgments(ranking, relevance)

    k = _positive_int("k", k)
    relevant = {document_id for document_id, grade in relevance.items() if int(grade) > 0}
    if not relevant:
        raise ValueError("relevance must contain at least one positive grade")
    return len(set(ranking[:k]) & relevant) / len(relevant)

def ndcg_at_k(ranking: Sequence[str], relevance: Mapping[str, int], k: int) -> float:
    """Return nDCG@k with gains 2^grade-1 and log2 discount."""
    _validate_judgments(ranking, relevance)

    k = _positive_int("k", k)
    grades = [int(grade) for grade in relevance.values()]
    if any(grade < 0 for grade in grades):
        raise ValueError("relevance grades must be non-negative")
    if not any(grade > 0 for grade in grades):
        raise ValueError("relevance must contain at least one positive grade")
    dcg = sum(
        (2 ** int(relevance.get(document_id, 0)) - 1.0) / math.log2(rank + 1.0)
        for rank, document_id in enumerate(ranking[:k], start=1)
    )
    ideal = sorted(grades, reverse=True)[:k]
    idcg = sum((2**grade - 1.0) / math.log2(rank + 1.0) for rank, grade in enumerate(ideal, start=1))
    return dcg / idcg

def evaluate_rankings(
    rankings: Mapping[str, Sequence[str]],
    query_records: Sequence[Mapping[str, object]],
    *,
    k: int = 5,
) -> dict[str, float]:
    """Aggregate MRR, Recall@k, and nDCG@k over one frozen query set."""

    k = _positive_int("k", k)
    if not query_records or len({r["query_id"] for r in query_records}) != len(query_records):
        raise ValueError("queries must be nonempty with unique IDs")
    if len(rankings) != len(query_records):
        raise ValueError("rankings must contain exactly one entry per query")
    reciprocal_ranks: list[float] = []
    recalls: list[float] = []
    ndcgs: list[float] = []
    for record in query_records:
        query_id = str(record["query_id"])
        if query_id not in rankings:
            raise KeyError(f"missing ranking for {query_id}")
        relevance = dict(record["relevance"])
        ranking = tuple(rankings[query_id])
        if len(ranking) != len(set(ranking)):
            raise ValueError(f"ranking for {query_id} contains duplicates")
        reciprocal_ranks.append(reciprocal_rank(ranking, relevance))
        recalls.append(recall_at_k(ranking, relevance, k))
        ndcgs.append(ndcg_at_k(ranking, relevance, k))
    return {
        "mrr": float(np.mean(reciprocal_ranks)),
        f"recall_at_{k}": float(np.mean(recalls)),
        f"ndcg_at_{k}": float(np.mean(ndcgs)),
    }
texts = ["TS-999 update fails after version 4.7.3 restart",
         "TS-998 update fails after version 4.7.3 restart",
         "Restart clears the cache after an update",
         "An update changes the search index"]
ids = ["error999","error998","cache","index"]
query = "TS-999 update"
bm = BM25Index(ids,texts); ls = LSAIndex(ids,texts,dimension=2)
branches = {"bm25":[r.document_id for r in bm.search(query)],
            "lsa":[r.document_id for r in ls.search(query)]}
fused = reciprocal_rank_fusion(branches)
records = {}
for depth in (1,2,4):
    head = [r.document_id for r in fused[:depth]]
    ranked = rerank_candidates(query,head,dict(zip(ids,texts)))
    records[depth] = {"ranking":[r.document_id for r in ranked], "pairs_scored":len(ranked)}
qrels = {"error999":3}
for depth,row in records.items():
    row["recall_at_1"] = recall_at_k(row["ranking"],qrels,1)
assert sum(row["pairs_scored"] for row in records.values()) == 7
print(json.dumps({"branches":branches,"stored_results":records},indent=2))
# Work is counted as pairs, not invented milliseconds.
plt.rcParams.update({"font.family":"DejaVu Sans","font.size":12,
    "axes.titlesize":17,"axes.labelsize":13,"xtick.labelsize":11,"ytick.labelsize":11})
plt.figure(figsize=(11,5))
plt.plot(list(records),[r["recall_at_1"] for r in records.values()],"o-")
plt.xlabel("Оценённые пары" if DEMO_LANGUAGE=="ru" else "Scored pairs")
plt.ylabel("Recall@1");plt.ylim(-.05,1.1);plt.xticks([1,2,4]);plt.tight_layout();plt.show()

Сопоставьте сохранённые ранги и Recall@1 с числом пар. Расширение списка полезно только тогда, когда оно возвращает нужного кандидата и оценка поднимает его выше остальных. Реранкер не может выбрать документ, которого не получил. Ровная кривая также содержательна: дополнительная работа не обязана менять ответ.

Во втором опыте первые строки матриц кодируют разное окружение, а две оставшиеся строки совпадают. Если сначала отрезать окружение, внимание получает одинаковые входы. Если применить внимание ко всему документу и только затем усреднить нужные строки, фрагмент может сохранить информацию об исключённой строке. Проследите, какие веса внимания переносят этот сигнал.

In [ ]:
# An exact attention operation with designed features, not a pretrained language model.
# First row = antecedent; remaining two rows form the chunk to retrieve.
def contextualize(x):
    logits = x @ x.T / np.sqrt(x.shape[1])
    logits -= logits.max(axis=1,keepdims=True)
    weights = np.exp(logits)
    weights /= weights.sum(axis=1,keepdims=True)
    return weights @ x

river = np.array([[2.,0.],[0.,1.],[0.,1.]])
finance = np.array([[0.,2.],[0.,1.],[0.,1.]])
early = [contextualize(x[1:]).mean(axis=0) for x in (river,finance)]
late = [contextualize(x)[1:].mean(axis=0) for x in (river,finance)]
assert np.array_equal(early[0],early[1])
assert not np.allclose(late[0],late[1])
print({"early_vectors":[x.tolist() for x in early],"late_vectors":[x.tolist() for x in late]})
# Changing only the excluded antecedent changes the pooled chunk after full attention.
assert late[0][0] > late[1][0]

При раннем разбиении оба вектора совпали; при позднем — различаются. Изменилась только строка вне выбранного фрагмента, поэтому опыт показывает перенос контекста через внимание. Специально заданные координаты не являются обученными языковыми представлениями: результат не доказывает понимание анафоры или преимущество на реальных документах.

## Переход к полному проекту

Один запрос служит диагностикой, а не тестовым распределением. Для проекта соберите отдельные запросы разработки и проверки, обучите или выберите эмбеддер без подбора по тесту и измерьте реальную задержку. Второй опыт показывает оператор, а не понимание местоимений: веса не обучались на языке. Для задачи У5 потребуются длинноконтекстный энкодер, реальные пары с анафорами и отдельный опыт с ColPali / OCR на одинаковых страницах; эта маленькая тетрадь не выполняет их и не заявляет превосходство визуального поиска.

### Минимальный отчёт

1. Версии данных, модели, кода и seed.
2. Основная метрика, а также качество, стоимость, задержка и память там, где это применимо.
3. Базовая линия, одинаковый бюджет и хотя бы один отрицательный контроль.
4. Неудачные прогоны и границы обоснованных выводов.